# 交互模式

来源:
- https://docs.langchain.com/mcp/frontend/branching-chat
- https://docs.langchain.com/mcp/frontend/message-queues
- https://docs.langchain.com/mcp/frontend/tool-calling
- https://docs.langchain.com/mcp/frontend/human-in-the-loop

本笔记覆盖前端交互的四种模式：分支聊天、消息队列、工具调用UI交互、人机协同审批UI。

---
## 1. 分支聊天 (Branching Chat)

分支聊天允许用户从任意消息点创建新的对话分支，探索不同的回复路径。

### 概念

```
原始对话                       分支
┌────────────────┐          ┌────────────────┐
│ 用户: 问题A    │          │ 用户: 问题A    │
├────────────────┤          ├────────────────┤
│ AI: 回答1      │          │ AI: 回答1      │
├────────────────┤          ├────────────────┤
│ 用户: 追问B    │          │ 用户: 追问B(不同)│
├────────────────┤          ├────────────────┤
│ AI: 回答2      │   ──→    │ AI: 回答2'     │
└────────────────┘          └────────────────┘
        │
        ▼ 在"回答1"处分叉
   ┌────────────┐ 
   │ 分支: ...  │
   └────────────┘
```

### React 实现

```tsx
// TypeScript React
import { useState, useCallback } from "react";

interface Branch {
  id: string;
  parentId: string | null;
  messages: Message[];
  createdAt: Date;
}

function useBranchingChat() {
  const [branches, setBranches] = useState<Branch[]>([
    { id: "main", parentId: null, messages: [], createdAt: new Date() }
  ]);
  const [activeBranchId, setActiveBranchId] = useState("main");

  const activeBranch = branches.find(b => b.id === activeBranchId)!;

  // 在指定消息位置创建分支
  const createBranch = useCallback((messageIndex: number) => {
    const newId = `branch-${Date.now()}`;
    const parentMessages = activeBranch.messages.slice(0, messageIndex + 1);
    
    setBranches(prev => [...prev, {
      id: newId,
      parentId: activeBranchId,
      messages: parentMessages,
      createdAt: new Date()
    }]);
    setActiveBranchId(newId);
  }, [activeBranch]);

  const switchBranch = (branchId: string) => {
    setActiveBranchId(branchId);
  };

  return {
    branches,
    activeBranch,
    activeBranchId,
    createBranch,
    switchBranch,
  };
}

// UI 组件
function BranchingChatInterface() {
  const {
    branches,
    activeBranch,
    activeBranchId,
    createBranch,
    switchBranch,
  } = useBranchingChat();

  return (
    <div className="branching-chat">
      {/* 分支导航 */}
      <div className="branch-tabs">
        {branches.map(branch => (
          <button
            key={branch.id}
            className={`branch-tab ${branch.id === activeBranchId ? "active" : ""}`}
            onClick={() => switchBranch(branch.id)}
          >
            {branch.id === "main" ? "主对话" : `分支 ${branch.id.slice(-4)}`}
          </button>
        ))}
      </div>

      {/* 消息列表 + 分支按钮 */}
      <div className="messages">
        {activeBranch.messages.map((msg, i) => (
          <div key={i} className="message-row">
            <MessageBubble message={msg} />
            <button
              className="branch-button"
              onClick={() => createBranch(i)}
              title="从此处创建分支"
            >
              🌿
            </button>
          </div>
        ))}
      </div>

      <MessageInput />
    </div>
  );
}
```

---
## 2. 消息队列 (Message Queues)

消息队列用于管理并发消息处理和事件排序。

### 核心概念

| 概念 | 说明 |
|------|------|
| **队列** | 按序处理的消息缓存 |
| **优先级** | 高优先级消息先处理 |
| **去重** | 防止重复处理同一消息 |
| **重试** | 失败消息自动重试 |

### React 实现

```tsx
// TypeScript React
import { useState, useCallback, useRef } from "react";

type TaskPriority = "high" | "normal" | "low";

interface QueueTask {
  id: string;
  type: "message" | "tool_call" | "interrupt_response";
  payload: any;
  priority: TaskPriority;
  retries: number;
}

function useMessageQueue() {
  const [queue, setQueue] = useState<QueueTask[]>([]);
  const [processing, setProcessing] = useState(false);
  const queueRef = useRef<QueueTask[]>([]);

  const enqueue = useCallback((task: QueueTask) => {
    queueRef.current = [...queueRef.current, task].sort((a, b) => {
      const priority: Record<TaskPriority, number> = {
        high: 0, normal: 1, low: 2
      };
      return priority[a.priority] - priority[b.priority];
    });
    setQueue([...queueRef.current]);
    processQueue();
  }, []);

  const processQueue = useCallback(async () => {
    if (processing || queueRef.current.length === 0) return;
    setProcessing(true);

    const task = queueRef.current.shift()!;
    setQueue([...queueRef.current]);

    try {
      await processTask(task);
    } catch (error) {
      if (task.retries < 3) {
        enqueue({ ...task, retries: task.retries + 1 });
      }
    }

    setProcessing(false);
    processQueue();  // 处理下一个
  }, [processing, enqueue]);

  // 去重
  const isDuplicate = (taskId: string): boolean => {
    return queueRef.current.some(t => t.id === taskId);
  };

  return { queue, enqueue, processing, isDuplicate };
}

// 使用
function ChatWithQueue() {
  const { enqueue, processing } = useMessageQueue();

  const handleSubmit = (message: string) => {
    enqueue({
      id: `msg-${Date.now()}`,
      type: "message",
      payload: { content: message },
      priority: "normal",
      retries: 0,
    });
  };

  return (
    <div className="chat">
      {processing && <div className="queue-indicator">处理队列中...</div>}
      <MessageInput onSubmit={handleSubmit} />
    </div>
  );
}
```

---
## 3. 工具调用 UI (Tool Calling)

工具调用UI展示Agent调用工具的过程、参数和结果。

### 工具调用生命周期

```
┌───────────┐    ┌───────────┐    ┌───────────┐    ┌───────────┐
│ 决定调用  │ → │ 请求中   │ → │ 执行中   │ → │ 完成     │
│ (pending) │   │ (loading) │   │ (running) │   │ (done)    │
└───────────┘    └───────────┘    └───────────┘    └───────────┘
                                          │
                                          ▼
                                     ┌───────────┐
                                     │ 失败      │
                                     │ (error)   │
                                     └───────────┘
```

### React 组件

```tsx
// TypeScript React
import React from "react";

interface ToolCall {
  id: string;
  name: string;
  args: Record<string, any>;
  status: "pending" | "loading" | "running" | "done" | "error";
  result?: any;
  error?: string;
  duration?: number;  // 毫秒
}

function ToolCallCard({ toolCall }: { toolCall: ToolCall }) {
  const statusColors = {
    pending: "gray",
    loading: "blue",
    running: "orange",
    done: "green",
    error: "red",
  };

  const statusIcons = {
    pending: "⏳",
    loading: "🔄",
    running: "⚙️",
    done: "✅",
    error: "❌",
  };

  return (
    <div className={`tool-call-card status-${toolCall.status}`}>
      <div className="tool-call-header">
        <span className="tool-status-icon">
          {statusIcons[toolCall.status]}
        </span>
        <span className="tool-name">{toolCall.name}</span>
        {toolCall.duration && (
          <span className="tool-duration">{toolCall.duration}ms</span>
        )}
      </div>

      <div className="tool-args">
        {Object.entries(toolCall.args).map(([key, value]) => (
          <div key={key} className="arg-row">
            <span className="arg-key">{key}:</span>
            <code className="arg-value">
              {typeof value === "object"
                ? JSON.stringify(value)
                : String(value)}
            </code>
          </div>
        ))}
      </div>

      {toolCall.status === "running" && (
        <div className="tool-loading-bar">
          <div className="loading-animation" />
        </div>
      )}

      {toolCall.status === "done" && toolCall.result && (
        <div className="tool-result">
          <pre>{JSON.stringify(toolCall.result, null, 2)}</pre>
        </div>
      )}

      {toolCall.status === "error" && (
        <div className="tool-error">
          <span>❌ {toolCall.error}</span>
          <button onClick={() => retry(toolCall.id)}>重试</button>
        </div>
      )}
    </div>
  );
}

// 工具调用列表
function ToolCallList({ toolCalls }: { toolCalls: ToolCall[] }) {
  return (
    <div className="tool-call-list">
      {toolCalls.map(tc => (
        <ToolCallCard key={tc.id} toolCall={tc} />
      ))}
    </div>
  );
}
```

---
## 4. 人机协同审批 UI (Human-in-the-Loop)

HITL UI在Agent需要人工决策时展示审批界面。

### 审批流程

```
[Agent] 需要审批
    │
    ▼
[前端] 显示审批对话框
    │
    ├── [批准] → Agent继续执行
    ├── [拒绝] → Agent回退/修改
    ├── [修改] → 编辑参数后批准
    └── [暂停] → 挂起，稍后处理
```

### React 审批组件

```tsx
// TypeScript React
import React, { useState } from "react";
import { useStream } from "@langchain/langgraph-sdk/react";

interface InterruptData {
  type: string;
  prompt: string;
  data: {
    action: string;
    args: Record<string, any>;
    reasoning: string;
  };
  options: string[];
}

// 审批对话框
function ApprovalDialog({ interrupt, onResolve }: {
  interrupt: InterruptData;
  onResolve: (decision: any) => void;
}) {
  const [showDetails, setShowDetails] = useState(false);
  const [modifiedArgs, setModifiedArgs] = useState(interrupt.data.args);
  const [reason, setReason] = useState("");

  return (
    <div className="approval-overlay">
      <div className="approval-dialog">
        <h3>🔐 需要审批</h3>
        <p className="approval-prompt">{interrupt.prompt}</p>

        <div className="approval-summary">
          <div className="info-row">
            <span className="label">操作:</span>
            <span className="value">{interrupt.data.action}</span>
          </div>
          <div className="info-row">
            <span className="label">Agent推理:</span>
            <span className="value">{interrupt.data.reasoning}</span>
          </div>
        </div>

        <button onClick={() => setShowDetails(!showDetails)}>
          {showDetails ? "收起参数" : "查看参数详情"}
        </button>

        {showDetails && (
          <div className="args-editor">
            {Object.entries(modifiedArgs).map(([key, value]) => (
              <div key={key} className="arg-edit-row">
                <label>{key}</label>
                <input
                  value={String(value)}
                  onChange={(e) => setModifiedArgs(prev => ({
                    ...prev, [key]: e.target.value
                  }))}
                />
              </div>
            ))}
          </div>
        )}

        <div className="reason-input">
          <label>理由 (可选):</label>
          <textarea
            value={reason}
            onChange={(e) => setReason(e.target.value)}
            placeholder="说明批准或拒绝的理由..."
          />
        </div>

        <div className="approval-actions">
          <button
            className="btn-approve"
            onClick={() => onResolve({ decision: "approve", reason })}
          >
            ✅ 批准
          </button>
          <button
            className="btn-modify"
            onClick={() => onResolve({
              decision: "modify",
              modified_args: modifiedArgs,
              reason
            })}
          >
            ✏️ 修改后批准
          </button>
          <button
            className="btn-reject"
            onClick={() => onResolve({ decision: "reject", reason })}
          >
            ❌ 拒绝
          </button>
        </div>
      </div>
    </div>
  );
}

// 集成到聊天界面
function ChatWithApproval() {
  const { messages, submit, interrupts, isLoading } = useStream({
    apiUrl: "http://localhost:8123",
    assistantId: "approval_agent",
  });

  const handleApproval = async (decision: any) => {
    await submit(Command.resume(decision));
  };

  return (
    <div className="chat">
      {/* 审批对话框 */}
      {interrupts.length > 0 && (
        <ApprovalDialog
          interrupt={interrupts[interrupts.length - 1]}
          onResolve={handleApproval}
        />
      )}

      {/* 消息列表 */}
      <MessageList messages={messages} />

      {/* 加载指示器 */}
      {isLoading && <ThinkingIndicator />}

      {/* 输入框 */}
      <MessageInput />
    </div>
  );
}
```

---
## 5. 交互模式对比

| 模式 | 核心问题 | 实现复杂度 | 用户体验提升 |
|------|----------|------------|-------------|
| 分支聊天 | 探索不同回复路径 | 中 | 高 (灵活探索) |
| 消息队列 | 并发消息顺序处理 | 低 | 中 (防止混乱) |
| 工具调用UI | 工具执行过程可视化 | 中 | 高 (透明度) |
| HITL审批 | 安全关键决策人工参与 | 中高 | 高 (安全信任) |

## 6. 参考链接

- [Branching Chat](https://docs.langchain.com/mcp/frontend/branching-chat)
- [Message Queues](https://docs.langchain.com/mcp/frontend/message-queues)
- [Tool Calling](https://docs.langchain.com/mcp/frontend/tool-calling)
- [Human-in-the-Loop](https://docs.langchain.com/mcp/frontend/human-in-the-loop)
- [LangGraph SDK React](https://www.npmjs.com/package/@langchain/langgraph-sdk)
- [React useStream Hook](https://langchain-ai.github.io/langgraph/how-tos/react-stream/)